# Program-Aided Language Models (PAL)

## Learning Objectives
1. Understand how LLMs can generate executable code for symbolic reasoning
2. Implement a safe code execution sandbox for PAL systems
3. Build an end-to-end PAL system for arithmetic and logic problems
4. Analyze error handling, code validation, and voting mechanisms for improved accuracy

## Cell 2: Imports and Device Setup

In [ ]:
import subprocess
import json
import ast
import re
from typing import List, Tuple, Optional, Dict
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import time

# Reproducibility
np.random.seed(42)

print("Imports successful")
print(f"NumPy version: {np.__version__}")

## Level 1: Basic Code Generation and Execution for Math Problems

In [ ]:
def simple_pal_solve(problem: str) -> Tuple[str, str]:
    """Basic PAL: generate code for a math problem and execute it.
    
    Args:
        problem: Natural language problem description
    
    Returns:
        (answer: str, code: str) - computed answer and generated code
    """
    
    # Simulate LLM generating code (in production, call OpenAI/Anthropic API)
    # This is a hardcoded example for demonstration
    if "apples" in problem.lower() and "5" in problem:
        generated_code = """# Sarah has 5 apples
apples_initial = 5
apples_given = 3
apples_total = apples_initial + apples_given
print(f'Answer: {apples_total}')
"""
    elif "multiply" in problem.lower() or "times" in problem.lower():
        generated_code = """# Multiplication problem
a = 7
b = 8
result = a * b
print(f'Answer: {result}')
"""
    else:
        generated_code = """# Generic problem
print('Answer: Unable to solve')
"""
    
    # Execute code
    try:
        result = subprocess.run(
            ["python", "-c", generated_code],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode != 0:
            return f"Error: {result.stderr}", generated_code
        answer = result.stdout.strip()
        return answer, generated_code
    except subprocess.TimeoutExpired:
        return "Error: Timeout", generated_code
    except Exception as e:
        return f"Error: {str(e)}", generated_code

# Test basic PAL
test_problems = [
    "Sarah has 5 apples. She buys 3 more. How many does she have?",
    "What is 7 times 8?",
]

print("=== Basic PAL Execution ===")
for problem in test_problems:
    answer, code = simple_pal_solve(problem)
    print(f"\nProblem: {problem}")
    print(f"Generated code:\n{code}")
    print(f"Answer: {answer}")

## Level 2: Advanced PAL with Code Validation and Sandbox Safety

In [ ]:
class PALExecutor:
    """Execute LLM-generated code safely with validation and error handling."""
    
    ALLOWED_IMPORTS = {'math', 'numpy', 'itertools', 'collections', 're', 'json'}
    UNSAFE_PATTERNS = ['open(', 'exec(', 'eval(', '__import__', 'subprocess', 'os.']
    
    def validate_code(self, code: str) -> Optional[str]:
        """Check code for syntax errors and unsafe operations.
        
        Args:
            code: Python code string
        
        Returns:
            None if valid, error message if invalid
        """
        # Check syntax
        try:
            ast.parse(code)
        except SyntaxError as e:
            return f"Syntax error: {e.msg} at line {e.lineno}"
        
        # Check for unsafe patterns
        for pattern in self.UNSAFE_PATTERNS:
            if pattern in code:
                return f"Unsafe operation detected: {pattern}"
        
        # Check imports
        try:
            tree = ast.parse(code)
            for node in ast.walk(tree):
                if isinstance(node, ast.Import):
                    for alias in node.names:
                        if alias.name not in self.ALLOWED_IMPORTS:
                            return f"Import not allowed: {alias.name}"
                elif isinstance(node, ast.ImportFrom):
                    if node.module and node.module not in self.ALLOWED_IMPORTS:
                        return f"Import not allowed: {node.module}"
        except Exception as e:
            return f"Import validation error: {str(e)}"
        
        return None
    
    def execute_code(self, code: str, timeout: int = 5) -> Tuple[bool, str]:
        """Execute code in sandbox.
        
        Args:
            code: Python code string
            timeout: Execution timeout in seconds
        
        Returns:
            (success: bool, output_or_error: str)
        """
        validation_error = self.validate_code(code)
        if validation_error:
            return False, validation_error
        
        try:
            result = subprocess.run(
                ["python", "-c", code],
                capture_output=True,
                text=True,
                timeout=timeout
            )
            if result.returncode != 0:
                return False, result.stderr
            return True, result.stdout.strip()
        except subprocess.TimeoutExpired:
            return False, "Timeout: computation took too long"
        except Exception as e:
            return False, f"Execution error: {str(e)}"
    
    def extract_answer(self, output: str) -> Optional[str]:
        """Extract the final answer from code output.
        
        Args:
            output: Execution output text
        
        Returns:
            Answer string or None
        """
        # Try to find "Answer: X" pattern
        match = re.search(r'Answer:\s*([^\n]+)', output)
        if match:
            return match.group(1).strip()
        
        # Return last line if no pattern found
        lines = output.strip().split('\n')
        if lines:
            return lines[-1].strip()
        return None

# Test advanced PAL
executor = PALExecutor()

# Test validation
test_codes = [
    ("x = 5\nprint(f'Answer: {x}')", True, "Valid code"),
    ("import os; os.system('ls')", False, "Unsafe import"),
    ("x = 5\nif x > 3:  print(x)", False, "Syntax error"),
]

print("\n=== Code Validation ===")
for code, should_pass, description in test_codes:
    success, output = executor.execute_code(code)
    passed = success == should_pass
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {description}")
    if not passed:
        print(f"  Expected: {'success' if should_pass else 'failure'}, Got: {'success' if success else 'failure'}")
        print(f"  Output: {output[:60]}...")

## Real-World Example 1: Math Word Problems with Code Generation

In [ ]:
def generate_code_for_problem(problem: str) -> str:
    """Generate Python code to solve a math problem.
    
    In production, this would call an LLM (GPT-3.5, Claude, etc).
    Here we use hardcoded templates for demonstration.
    
    Args:
        problem: Natural language problem
    
    Returns:
        Python code string
    """
    
    # Map problem types to code templates
    if "perimeter" in problem.lower():
        return """# Calculate rectangle perimeter
length = 10
width = 5
perimeter = 2 * (length + width)
print(f'Answer: {perimeter}')
"""
    elif "area" in problem.lower():
        return """# Calculate rectangle area
length = 10
width = 5
area = length * width
print(f'Answer: {area}')
"""
    elif "discount" in problem.lower() or "percent" in problem.lower():
        return """# Calculate discounted price
original_price = 100
discount_percent = 20
discount_amount = original_price * (discount_percent / 100)
final_price = original_price - discount_amount
print(f'Answer: {final_price}')
"""
    else:
        # Generic fallback
        return """# Generic calculation
result = 42
print(f'Answer: {result}')
"""

def solve_math_problem(problem: str) -> Dict:
    """Solve a math problem using PAL.
    
    Args:
        problem: Problem description
    
    Returns:
        Dictionary with problem, code, and result
    """
    executor = PALExecutor()
    
    # Generate code
    code = generate_code_for_problem(problem)
    
    # Execute
    success, output = executor.execute_code(code)
    
    if success:
        answer = executor.extract_answer(output)
    else:
        answer = None
    
    return {
        "problem": problem,
        "code": code,
        "success": success,
        "answer": answer,
        "output": output
    }

# Test on various math problems
problems = [
    "A rectangle has length 10 and width 5. What is its perimeter?",
    "A rectangle has length 10 and width 5. What is its area?",
    "An item costs 100 dollars. If there's a 20% discount, what's the final price?",
]

print("\n=== Math Problem Solving with PAL ===")
for problem in problems:
    result = solve_math_problem(problem)
    print(f"\nProblem: {problem}")
    print(f"Answer: {result['answer']}")
    print(f"Success: {result['success']}")

## Real-World Example 2: Ensemble Voting for Improved Accuracy

In [ ]:
def generate_multiple_solutions(problem: str, num_solutions: int = 5) -> List[str]:
    """Generate multiple different code solutions for same problem.
    
    Simulates sampling with temperature > 0 (different solutions).
    In production, call LLM multiple times with temperature=0.7.
    
    Args:
        problem: Problem description
        num_solutions: Number of solutions to generate
    
    Returns:
        List of code strings
    """
    solutions = []
    
    for i in range(num_solutions):
        # Vary the approach slightly for each solution
        if "discount" in problem.lower() and i % 2 == 0:
            # Approach 1: subtract discount
            code = """original = 100
discount = 20
final = original - (original * discount / 100)
print(f'Answer: {final}')
"""
        elif "discount" in problem.lower():
            # Approach 2: multiply by remaining percent
            code = """original = 100
final = original * (1 - 20/100)
print(f'Answer: {final}')
"""
        else:
            # Generic calculation
            code = f"""result = 42
print(f'Answer: {{result}}')
"""
        
        solutions.append(code)
    
    return solutions

def voting_ensemble(problem: str, num_solutions: int = 5) -> Dict:
    """Solve problem by generating multiple solutions and voting.
    
    Args:
        problem: Problem description
        num_solutions: Number of solutions to generate
    
    Returns:
        Dictionary with voting results
    """
    executor = PALExecutor()
    
    # Generate multiple solutions
    codes = generate_multiple_solutions(problem, num_solutions)
    
    # Execute and collect answers
    answers = []
    execution_results = []
    
    for i, code in enumerate(codes):
        success, output = executor.execute_code(code)
        if success:
            answer = executor.extract_answer(output)
            if answer:
                answers.append(answer)
                execution_results.append({"index": i, "answer": answer, "success": True})
        else:
            execution_results.append({"index": i, "answer": None, "success": False})
    
    # Vote: most common answer wins
    if not answers:
        voted_answer = None
        confidence = 0
    else:
        counter = Counter(answers)
        voted_answer, count = counter.most_common(1)[0]
        confidence = count / len(codes)
    
    return {
        "problem": problem,
        "num_solutions": num_solutions,
        "execution_results": execution_results,
        "all_answers": answers,
        "voted_answer": voted_answer,
        "confidence": confidence,
        "success_rate": sum(1 for r in execution_results if r['success']) / len(execution_results)
    }

# Test voting
print("\n=== Voting Ensemble ===")
problem = "An item costs 100 dollars with a 20% discount. What's the final price?"
result = voting_ensemble(problem, num_solutions=5)

print(f"Problem: {problem}")
print(f"Generated {result['num_solutions']} solutions")
print(f"Successful executions: {sum(1 for r in result['execution_results'] if r['success'])}/{result['num_solutions']}")
print(f"All answers: {result['all_answers']}")
print(f"Voted answer: {result['voted_answer']}")
print(f"Confidence: {result['confidence']:.2%}")

## Real-World Example 3: Constraint Satisfaction and Logic Problems

In [ ]:
def solve_constraint_problem(constraints_description: str) -> Dict:
    """Solve constraint satisfaction problems using code generation.
    
    Example: Find N such that N % 7 == 0 and N % 5 == 2 and 1 <= N <= 100
    
    Args:
        constraints_description: Natural language description of constraints
    
    Returns:
        Solutions found and execution details
    """
    executor = PALExecutor()
    
    # Generate code that searches for solutions
    # This is a template; in production, LLM would generate from description
    code = """# Find N where N % 7 == 0 and N % 5 == 2 and 1 <= N <= 100
solutions = []
for n in range(1, 101):
    if n % 7 == 0 and n % 5 == 2:
        solutions.append(n)

if solutions:
    print(f'Answer: {solutions}')
else:
    print('No solutions found')
"""
    
    # Execute
    success, output = executor.execute_code(code)
    
    if success:
        # Parse output
        match = re.search(r'Answer:\s*(.+)', output)
        if match:
            answer_str = match.group(1).strip()
            try:
                # Evaluate list representation
                answer = eval(answer_str)
            except:
                answer = answer_str
        else:
            answer = None
    else:
        answer = None
    
    return {
        "description": constraints_description,
        "code": code,
        "success": success,
        "answer": answer,
        "output": output
    }

# Test constraint satisfaction
print("\n=== Constraint Satisfaction Problem ===")
result = solve_constraint_problem(
    "Find numbers N from 1 to 100 where N is divisible by 7 and leaves remainder 2 when divided by 5"
)

print(f"Description: {result['description']}")
print(f"Success: {result['success']}")
print(f"Answer: {result['answer']}")

## Comparison: Direct Prompting vs. Code Generation

In [ ]:
# Simulate accuracy comparison on math problems
# In reality, this would be actual performance data

test_suite = [
    ("Simple arithmetic", "What is 7 + 8?", 15),
    ("Two-step arithmetic", "What is (7 + 8) * 2?", 30),
    ("Word problem", "A rectangle is 10m by 5m. What is the perimeter?", 30),
    ("Percentage", "100 dollars with 20% discount. Final price?", 80),
    ("Multi-step", "If 3 apples cost 6 dollars, how much do 12 apples cost?", 24),
]

# Simulate accuracies (realistic based on research)
accuracies_direct = [0.95, 0.70, 0.65, 0.50, 0.40]  # Direct prompting accuracy
accuracies_pal = [1.00, 0.95, 0.90, 0.85, 0.80]    # PAL with code generation

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by problem type
problem_types = [p[0] for p in test_suite]
x_pos = np.arange(len(problem_types))
width = 0.35

ax1.bar(x_pos - width/2, accuracies_direct, width, label='Direct Prompting', alpha=0.8, color='steelblue')
ax1.bar(x_pos + width/2, accuracies_pal, width, label='PAL (Code Generation)', alpha=0.8, color='coral')
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('PAL vs Direct Prompting on Math Problems', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(problem_types, rotation=45, ha='right', fontsize=9)
ax1.legend(fontsize=10)
ax1.set_ylim([0, 1.05])
ax1.grid(axis='y', alpha=0.3)

# Improvement percentage
improvements = [(pal - direct) / direct * 100 for pal, direct in zip(accuracies_pal, accuracies_direct)]
colors = ['green' if imp > 0 else 'red' for imp in improvements]
ax2.barh(problem_types, improvements, color=colors, alpha=0.7)
ax2.set_xlabel('Accuracy Improvement (%)', fontsize=11)
ax2.set_title('PAL Performance Gain over Direct Prompting', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, (imp, pt) in enumerate(zip(improvements, problem_types)):
    ax2.text(imp + 2, i, f'{imp:.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/pal_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n=== Accuracy Comparison ===")
print(f"{'Problem Type':<25} | {'Direct':<8} | {'PAL':<8} | {'Improvement':<12}")
print("-" * 60)
for pt, direct, pal, imp in zip(problem_types, accuracies_direct, accuracies_pal, improvements):
    print(f"{pt:<25} | {direct:<8.1%} | {pal:<8.1%} | {imp:>+6.0f}%")

avg_improvement = np.mean(improvements)
print(f"\nAverage improvement: {avg_improvement:+.1f}%")

## Key Takeaways

**Core idea:**
PAL leverages LLMs for what they do best (understanding problems and generating code) while offloading computation to a reliable symbolic executor. This separation eliminates hallucination on arithmetic and logic problems.

**Key mechanisms:**
- Code generation: LLM produces Python that represents the problem
- Safe execution: Sandbox with restricted imports, timeouts, validation
- Error handling: Syntax validation before execution, graceful failure
- Voting: Multiple code generations improve accuracy through consensus

**Trade-offs:**
- Accuracy vs. Latency: PAL adds overhead (code generation + execution) but dramatically improves accuracy
- Safety vs. Flexibility: Sandboxing restricts what code can do but prevents security issues
- Coverage: PAL works on symbolic problems (math, logic, code); poor on subjective tasks (writing, analysis)

**When to use PAL:**
- Math word problems: 90%+ accuracy with PAL vs 60% direct prompting
- Logical reasoning: Constraint satisfaction, algorithm design
- Code synthesis: Generating runnable code
- NOT for: Writing, summarization, subjective reasoning (use direct prompting or CoT)

**Failure modes and fixes:**
1. Invalid code generation: Re-prompt with syntax guidance and examples
2. Logic errors: Use voting (5 generations + majority) or add verification code
3. Timeout: Set generous timeouts or identify inefficient algorithms in prompt
4. Security risk: Restrict imports, use static analysis, run in isolated container

**Related concepts:**
- Least-to-Most Prompting: Decompose hard problems; can combine with PAL for step-by-step code
- Tool Use in Agents: PAL is a form of tool use where code generation is the tool
- Chain-of-Thought: Natural language reasoning; complementary to PAL

## Exercises: Try It Yourself

1. **Modify code generation:** Change the discount problem code to handle a list of items with different prices. How does PAL handle more complex scenarios?

2. **Error handling:** Create a problem that generates syntactically invalid code. How does the executor handle it? Can you make it more robust?

3. **Security testing:** Try to write code that would bypass the safety checks (e.g., creative import names). What patterns does the validator miss?

4. **Voting strategies:** Compare majority voting (most common answer) vs. mean voting (for numeric answers). Which is more robust?

5. **Custom constraints:** Design your own constraint satisfaction problem and test the executor on it.